In [ ]:
import sys, os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/joshuaswords/netflix-data-visualization/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_24_try_1.pickle

In [ ]:
%%PandasProfile
### cell 25 ###

# Filter for USA and India once
us_ind = df[df['first_country'].isin(['USA', 'India'])]

# Convert Modin to pandas if needed
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df = df._to_pandas()

# Compute raw counts by year and country in one groupby, only for the two countries
counts = (df[df['first_country'].isin(['USA', 'India'])]
            .groupby(['year_added', 'first_country'])
            .size()
            .unstack(fill_value=0))

# Combined counts and USA-only counts
combined = counts.sum(axis=1)
us_counts = counts['USA']
half_comb = combined / 2

# Build the final DataFrame with base, USA, India columns
data_sub = pd.DataFrame({
    'base': -half_comb,
    'USA': us_counts - half_comb,
    'India': half_comb
})

# Convert back to Modin if needed
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    df = pd.DataFrame(df)
    data_sub = pd.DataFrame(data_sub)